In [ ]:
%load_ext kedro.ipython

In [13]:
companies = catalog.load("companies")
companies.head()

[10/28/25 16:55:25] INFO     Loading data from companies (CSVDataset)...                        ]8;id=726299;file://c:\Users\sebastian.calderon\Documents\GitHub\work\kedro\venv\Lib\site-packages\kedro\io\data_catalog.py\data_catalog.py]8;;\:]8;id=222360;file://c:\Users\sebastian.calderon\Documents\GitHub\work\kedro\venv\Lib\site-packages\kedro\io\data_catalog.py#403\403]8;;\

,id,company_rating,company_location,total_fleet_count,iata_approved
0,3888,100%,Isle of Man,1.0,f
1,46728,100%,NaN,1.0,f
2,34618,38%,Isle of Man,1.0,f
3,28619,100%,Bosnia and Herzegovina,1.0,f
4,8240,NaN,Chile,1.0,t


In [14]:
companies["iata_approved"] = companies["iata_approved"] == "t"
companies["company_rating"] = companies["company_rating"].str.replace("%","").astype(float)
companies.head()

,id,company_rating,company_location,total_fleet_count,iata_approved
0,3888,100.0,Isle of Man,1.0,False
1,46728,100.0,NaN,1.0,False
2,34618,38.0,Isle of Man,1.0,False
3,28619,100.0,Bosnia and Herzegovina,1.0,False
4,8240,NaN,Chile,1.0,True


In [15]:
shuttles = catalog.load("shuttles")
shuttles.head()

[10/28/25 16:56:02] INFO     Loading data from shuttles (ExcelDataset)...                       ]8;id=746805;file://c:\Users\sebastian.calderon\Documents\GitHub\work\kedro\venv\Lib\site-packages\kedro\io\data_catalog.py\data_catalog.py]8;;\:]8;id=537710;file://c:\Users\sebastian.calderon\Documents\GitHub\work\kedro\venv\Lib\site-packages\kedro\io\data_catalog.py#403\403]8;;\

,id,shuttle_location,shuttle_type,engine_type,engine_vendor,engines,passenger_capacity,cancellation_policy,crew,d_check_complete,moon_clearance_complete,price,company_id
0,45163,Sao Tome and Principe,Type V5,Plasma,ThetaBase Services,2.0,4,moderate,2.0,f,f,"$1,715.0",32413
1,49438,Wallis and Futuna,Type V2,Plasma,ThetaBase Services,3.0,5,moderate,3.0,f,f,"$3,405.0",14122
2,10750,Niue,Type F5,Quantum,ThetaBase Services,1.0,2,strict,1.0,t,f,"$1,806.0",47761
3,4146,Malta,Type V2,Quantum,ThetaBase Services,1.0,2,moderate,1.0,f,f,"$1,676.0",26648
4,5067,Malta,Type V2,Plasma,ThetaBase Services,5.0,10,strict,5.0,f,f,"$4,718.0",26648


In [16]:
shuttles["d_check_complete"] = shuttles["d_check_complete"] == "t"
shuttles["moon_clearance_complete"] = shuttles["moon_clearance_complete"] == "t"
shuttles["price"] = shuttles["price"].str.replace("$","").str.replace(",","").astype(float)
shuttles.head()

,id,shuttle_location,shuttle_type,engine_type,engine_vendor,engines,passenger_capacity,cancellation_policy,crew,d_check_complete,moon_clearance_complete,price,company_id
0,45163,Sao Tome and Principe,Type V5,Plasma,ThetaBase Services,2.0,4,moderate,2.0,False,False,1715.0,32413
1,49438,Wallis and Futuna,Type V2,Plasma,ThetaBase Services,3.0,5,moderate,3.0,False,False,3405.0,14122
2,10750,Niue,Type F5,Quantum,ThetaBase Services,1.0,2,strict,1.0,True,False,1806.0,47761
3,4146,Malta,Type V2,Quantum,ThetaBase Services,1.0,2,moderate,1.0,False,False,1676.0,26648
4,5067,Malta,Type V2,Plasma,ThetaBase Services,5.0,10,strict,5.0,False,False,4718.0,26648


Ahora, se va a hacer lo mismo, pero adaptandolo para poder usar pipelines de kedro

In [21]:
import pandas as pd

def _is_true(x: pd.Series) -> pd.Series:
    return x == "t"

def _parse_percentage(x: pd.Series) -> pd.Series:
    x = x.str.replace("%", "")
    x = x.astype(float) / 100
    return x


def _parse_money(x: pd.Series) -> pd.Series:
    x = x.str.replace("$", "").str.replace(",", "")
    x = x.astype(float)
    return x

In [22]:
companies = catalog.load("companies")

companies["iata_approved"] = _is_true(companies["iata_approved"])
companies["company_rating"] = _parse_percentage(companies["company_rating"])
companies.dtypes

[10/28/25 17:33:36] INFO     Loading data from companies (CSVDataset)...                        ]8;id=801520;file://c:\Users\sebastian.calderon\Documents\GitHub\work\kedro\venv\Lib\site-packages\kedro\io\data_catalog.py\data_catalog.py]8;;\:]8;id=715454;file://c:\Users\sebastian.calderon\Documents\GitHub\work\kedro\venv\Lib\site-packages\kedro\io\data_catalog.py#403\403]8;;\


id                     int64
company_rating       float64
company_location      object
total_fleet_count    float64
iata_approved           bool
dtype: object

In [23]:
shuttles = catalog.load("shuttles")

shuttles["d_check_complete"] = _is_true(shuttles["d_check_complete"])
shuttles["moon_clearance_complete"] = _is_true(shuttles["moon_clearance_complete"])
shuttles["price"] = _parse_money(shuttles["price"])
shuttles.dtypes

[10/28/25 17:34:17] INFO     Loading data from shuttles (ExcelDataset)...                       ]8;id=519418;file://c:\Users\sebastian.calderon\Documents\GitHub\work\kedro\venv\Lib\site-packages\kedro\io\data_catalog.py\data_catalog.py]8;;\:]8;id=175097;file://c:\Users\sebastian.calderon\Documents\GitHub\work\kedro\venv\Lib\site-packages\kedro\io\data_catalog.py#403\403]8;;\


id                           int64
shuttle_location            object
shuttle_type                object
engine_type                 object
engine_vendor               object
engines                    float64
passenger_capacity           int64
cancellation_policy         object
crew                       float64
d_check_complete              bool
moon_clearance_complete       bool
price                      float64
company_id                   int64
dtype: object

In [ ]:
reviews = catalog.load("reviews")